# 05b · Diagnostic: is the low English accuracy caused by Qwen or the classifier?

In notebook 05:
- the frozen mBERT classifier (notebook 04) scored ~0.69 on the original Chinese posts
- but only ~0.30 on Qwen's English translations

Also, it predicted "neutral" for 51–61% of translations, but only ~5% of the posts are actually neutral.

There are two possible explanations:
1. Qwen's translations lose the emotion.
2. The classifier can't read emotion in this kind of English (for example because its English training data, GoEmotions, is 77% neutral).

To figure out which one, I run the classifiers on professional translations of the same kind of posts made by humans. HADQAET has social media posts with emotion labels, a Google Translate version, and human post-edited English translations (where a human judged the Google Translate version wasn't good enough).

All HADQAET posts were excluded from classifier training, so the classifiers have never seen them.

HADQAET has no neutral class, so every neutral prediction here is wrong. That makes "% predicted neutral" a direct measure of the problem.

I rewrote this notebook to check all three classifiers in one place:

| classifier | notebook | model folder |
|---|---|---|
| mBERT original | 04 | `models/mbert_emotion_6class/checkpoints/checkpoint-7134` |
| mBERT rebalanced | 04b | `models/mbert_emotion_6class_rebalanced/final` |
| XLM-R rebalanced | 04c | `models/xlmr_emotion_6class_rebalanced/final` |


## 0. Get the repo into Colab

This has to run FIRST, before anything that builds a file path. Otherwise the paths get built from `/content` before the repo exists and nothing is found.

If the repo is private, swap the clone line for the token version (see notebook 05b).

In [1]:
# Install Git LFS so the large model files download, src: https://huggingface.co/spaces/huggingface/HuggingDiscussions/discussions/5
!apt-get -qq install git-lfs
!git lfs install

# clone repo (skip if it's already here from earlier in this session)
import os
if not os.path.exists("/content/W266-slang-emotion-mt"):
    !git clone https://github.com/cierrajagan/W266-slang-emotion-mt.git

%cd /content/W266-slang-emotion-mt

# make sure the real model files download, not the tiny LFS placeholders
!git lfs pull

Selecting previously unselected package git-lfs.
(Reading database ... 127324 files and directories currently installed.)
Preparing to unpack .../git-lfs_3.4.1-1ubuntu0.4_amd64.deb ...
Unpacking git-lfs (3.4.1-1ubuntu0.4) ...
Setting up git-lfs (3.4.1-1ubuntu0.4) ...
Processing triggers for man-db (2.12.0-4build2) ...
Git LFS initialized.
Cloning into 'W266-slang-emotion-mt'...
remote: Enumerating objects: 162, done.
remote: Counting objects: 100% (162/162), done.
remote: Compressing objects: 100% (122/122), done.
remote: Total 162 (delta 61), reused 131 (delta 34), pack-reused 0 (from 0)
Receiving objects: 100% (162/162), 19.50 MiB | 8.92 MiB/s, done.
Resolving deltas: 100% (61/61), done.
/content/W266-slang-emotion-mt


The rebalanced models (04b, 04c) are saved to Google Drive for now (not GitHub yet). Run this to copy them into the repo folder. Will update this once I push to github

In [2]:
# Copy the rebalanced models from Google Drive (saved at the end of notebooks 04b and 04c)
from google.colab import drive
drive.mount("/content/drive")

!mkdir -p models
!cp -r "/content/drive/MyDrive/ISchool/MIDS/266/models/mbert_emotion_6class_rebalanced" models/
!cp -r "/content/drive/MyDrive/ISchool/MIDS/266/models/xlmr_emotion_6class_rebalanced" models/

Mounted at /content/drive


## 1. Setup

In [3]:
%pip install -q -U "transformers>=4.45,<6" scikit-learn

In [4]:
import os
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import accuracy_score, f1_score
from transformers import AutoModelForSequenceClassification, AutoTokenizer

In [5]:
# Folders (same logic as the other notebooks)
current_folder = os.getcwd()
if os.path.basename(current_folder) == "notebooks":
    project_root = os.path.dirname(current_folder)
else:
    project_root = current_folder

processed_folder = os.path.join(project_root, "data", "processed")
results_folder = os.path.join(project_root, "data", "results")
post_edits_path = os.path.join(processed_folder, "hadqaet_post_edits.csv")

# The three classifiers to check
classifiers = {
    "mBERT original (nb 04)": os.path.join(project_root, "models", "mbert_emotion_6class", "checkpoints", "checkpoint-7134"),
    "mBERT rebalanced (nb 04b)": os.path.join(project_root, "models", "mbert_emotion_6class_rebalanced", "final"),
    "XLM-R rebalanced (nb 04c)": os.path.join(project_root, "models", "xlmr_emotion_6class_rebalanced", "final"),
}

print("Project root:", project_root)
print("HADQAET post-edit file exists:", os.path.exists(post_edits_path))
for classifier_name in classifiers:
    print(classifier_name, "| folder exists:", os.path.exists(classifiers[classifier_name]))

Project root: /content/W266-slang-emotion-mt
HADQAET post-edit file exists: True
mBERT original (nb 04) | folder exists: True
mBERT rebalanced (nb 04b) | folder exists: True
XLM-R rebalanced (nb 04c) | folder exists: True


In [6]:
# Make sure using GPU
if torch.cuda.is_available():
    device = "cuda"
else:
    device = "cpu"
print("Running on:", device)

Running on: cuda


## 2. Load HADQAET and build the three versions of each post

For every HADQAET post we have:
- chinese: the original Weibo post (source)
- google_mt: Google Translate's English (MT)
- human_english: the human-quality English. This is the post-edit where annotators corrected an error; where annotators found no error, Google's translation was already judged correct, so we use it as-is.

We also make a smaller subset of only posts that have a post-edit, because there the Google and human versions are really different.

In [7]:
# labels in the same order as notebook 04 (the model's output order)
LABELS = ["joy", "anger", "sadness", "fear", "surprise", "neutral"]

In [8]:
hadqaet = pd.read_csv(post_edits_path)
print("HADQAET rows:", len(hadqaet))

# Clean the gold labels (strip spaces, lowercase)
gold_labels = []
for label in hadqaet["emotion_labels"]:
    gold_labels.append(str(label).strip().lower())
hadqaet["gold"] = gold_labels

# make sure every gold label is one of our 6 labels
unknown = set(hadqaet["gold"]) - set(LABELS)
print("Labels not in our set (should be empty):", unknown)

# Build the human-quality English column
human_english = []
has_post_edit = []
for index, row in hadqaet.iterrows():
    post_edit = row["post-edit"]
    if pd.isna(post_edit) or str(post_edit).strip() == "":
        # If no error was annotated, then Google's translation was correct
        human_english.append(str(row["MT"]))
        has_post_edit.append(False)
    else:
        human_english.append(str(post_edit))
        has_post_edit.append(True)

hadqaet["human_english"] = human_english
hadqaet["has_post_edit"] = has_post_edit

print("Num post-edit:", sum(has_post_edit))
print("Num correct Google MT:", len(hadqaet) - sum(has_post_edit))
print(hadqaet["gold"].value_counts())

HADQAET rows: 4038
Labels not in our set (should be empty): set()
Num post-edit: 2952
Num correct Google MT: 1086
gold
anger       1429
joy         1187
sadness      887
surprise     282
fear         253
Name: count, dtype: int64


## 3. Helper functions

In [9]:
# Helper function: predict an emotion for each text in a list
# texts = a list of strings
# batch_size = how many texts to send to the model at once
# Returns a list of predicted labels, e.g. ["anger", "neutral", ...]

def predict_emotions(texts, batch_size=32):
    predictions = []

    # Go through the texts one batch at a time
    for batch_start in range(0, len(texts), batch_size):
        batch_texts = texts[batch_start:batch_start + batch_size]

        # Turn the texts into tokens (same settings as notebooks 04 and 05: max 128 tokens)
        inputs = tokenizer(
            batch_texts,
            truncation=True,
            max_length=128,
            padding=True,
            return_tensors="pt",
        ).to(device)

        with torch.no_grad(): # torch.no_grad() = we're only predicting, so don't track gradients (faster), src: https://stackoverflow.com/questions/68579174/when-to-use-torch-no-grad-is-safe-in-forward-propagation-why-does-it-hurt-my
            logits = model(**inputs).logits

        # The highest score in each row is the predicted class
        predicted_ids = logits.argmax(dim=1).tolist()
        for predicted_id in predicted_ids:
            predictions.append(LABELS[predicted_id])

    return predictions

In [10]:
# Helper function: accuracy, macro-F1 and % predicted neutral

def score(gold, predicted):
    accuracy = accuracy_score(gold, predicted)

    # Macro-F1 over the 5 emotions that actually occur in HADQAET (no neutral gold labels)
    macro_f1 = f1_score(
        gold, predicted,
        labels=["joy", "anger", "sadness", "fear", "surprise"],
        average="macro",
        zero_division=0,
    )

    # count neutral predictions
    neutral_count = 0
    for label in predicted:
        if label == "neutral":
            neutral_count = neutral_count + 1
    percent_neutral = neutral_count / len(predicted) * 100

    return accuracy, macro_f1, percent_neutral

## 4. Run every classifier on all three versions

For each classifier: load it, predict on the Chinese, Google MT and human-quality English versions, and save the predictions as new columns (e.g. `mBERT original (nb 04) | Human-quality English`).

In [11]:
versions = {
    "Chinese original": "source",
    "Google MT English": "MT",
    "Human-quality English": "human_english",
}

classifiers_run = []   # the classifiers we actually found and ran

for classifier_name in classifiers:
    model_folder = classifiers[classifier_name]

    # skip classifiers whose folder isn't here
    if not os.path.exists(model_folder):
        print("SKIPPING", classifier_name, "- folder not found:", model_folder)
        continue

    print("\nLoading", classifier_name)
    tokenizer = AutoTokenizer.from_pretrained(model_folder)
    model = AutoModelForSequenceClassification.from_pretrained(model_folder)
    model.to(device)
    model.eval()

    # make sure the model's labels are in the same order as ours
    model_labels = []
    for i in range(model.config.num_labels):
        model_labels.append(model.config.id2label[i])
    assert model_labels == LABELS, "Label order doesn't match notebook 04!"

    for version_name in versions:
        text_column = versions[version_name]
        print("  classifying", version_name)
        hadqaet[classifier_name + " | " + version_name] = predict_emotions(hadqaet[text_column].astype(str).tolist())

    classifiers_run.append(classifier_name)

print("\nDone:", classifiers_run)


Loading mBERT original (nb 04)


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

  classifying Chinese original
  classifying Google MT English
  classifying Human-quality English

Loading mBERT rebalanced (nb 04b)


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

  classifying Chinese original
  classifying Google MT English
  classifying Human-quality English

Loading XLM-R rebalanced (nb 04c)


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

  classifying Chinese original
  classifying Google MT English
  classifying Human-quality English

Done: ['mBERT original (nb 04)', 'mBERT rebalanced (nb 04b)', 'XLM-R rebalanced (nb 04c)']


## 5. Results

In [12]:
# Score each classifier x version, on all posts and only posts with a post-edit
summary_rows = []
subsets = {
    "all HADQAET posts": hadqaet,
    "posts with a post-edit": hadqaet[hadqaet["has_post_edit"] == True],
}

for subset_name in subsets:
    subset = subsets[subset_name]
    for classifier_name in classifiers_run:
        for version_name in versions:
            prediction_column = classifier_name + " | " + version_name
            accuracy, macro_f1, percent_neutral = score(subset["gold"], subset[prediction_column])
            summary_rows.append({
                "posts": subset_name,
                "classifier": classifier_name,
                "text version": version_name,
                "n": len(subset),
                "accuracy": round(accuracy, 3),
                "macro_f1": round(macro_f1, 3),
                "% predicted neutral": round(percent_neutral, 1),
            })

summary = pd.DataFrame(summary_rows)

# The main table: all posts only
summary[summary["posts"] == "all HADQAET posts"]

,posts,classifier,text version,n,accuracy,macro_f1,% predicted neutral
0,all HADQAET posts,mBERT original (nb 04),Chinese original,4038,0.753,0.734,4.3
1,all HADQAET posts,mBERT original (nb 04),Google MT English,4038,0.335,0.467,55.3
2,all HADQAET posts,mBERT original (nb 04),Human-quality English,4038,0.378,0.499,49.9
3,all HADQAET posts,mBERT rebalanced (nb 04b),Chinese original,4038,0.749,0.732,4.7
4,all HADQAET posts,mBERT rebalanced (nb 04b),Google MT English,4038,0.507,0.549,24.0
5,all HADQAET posts,mBERT rebalanced (nb 04b),Human-quality English,4038,0.553,0.579,19.3
6,all HADQAET posts,XLM-R rebalanced (nb 04c),Chinese original,4038,0.790,0.767,3.3
7,all HADQAET posts,XLM-R rebalanced (nb 04c),Google MT English,4038,0.557,0.606,23.3
8,all HADQAET posts,XLM-R rebalanced (nb 04c),Human-quality English,4038,0.600,0.629,18.8


In [13]:
# What does each classifier predict for each version? (% of posts)

distribution_rows = []
for classifier_name in classifiers_run:
    for version_name in versions:
        prediction_column = classifier_name + " | " + version_name
        counts = hadqaet[prediction_column].value_counts(normalize=True) * 100
        row = {"classifier": classifier_name, "text version": version_name}
        for label in LABELS:
            if label in counts.index:
                row[label] = round(counts[label], 1)
            else:
                row[label] = 0.0
        distribution_rows.append(row)

# Add the true (gold) distribution for comparison
gold_counts = hadqaet["gold"].value_counts(normalize=True) * 100
gold_row = {"classifier": "GOLD (true labels)", "text version": ""}
for label in LABELS:
    if label in gold_counts.index:
        gold_row[label] = round(gold_counts[label], 1)
    else:
        gold_row[label] = 0.0
distribution_rows.append(gold_row)

pd.DataFrame(distribution_rows)

,classifier,text version,joy,anger,sadness,fear,surprise,neutral
0,mBERT original (nb 04),Chinese original,26.6,33.6,23.2,5.8,6.6,4.3
1,mBERT original (nb 04),Google MT English,12.7,8.3,13.9,7.4,2.4,55.3
2,mBERT original (nb 04),Human-quality English,13.2,11.4,14.9,7.5,3.2,49.9
3,mBERT rebalanced (nb 04b),Chinese original,26.4,33.9,22.6,6.0,6.4,4.7
4,mBERT rebalanced (nb 04b),Google MT English,21.2,15.1,24.1,10.5,5.1,24.0
5,mBERT rebalanced (nb 04b),Human-quality English,21.8,18.2,24.0,10.7,5.9,19.3
6,XLM-R rebalanced (nb 04c),Chinese original,27.2,33.9,21.9,6.5,7.2,3.3
7,XLM-R rebalanced (nb 04c),Google MT English,22.5,16.2,23.7,9.5,4.9,23.3
8,XLM-R rebalanced (nb 04c),Human-quality English,22.7,19.3,23.7,9.7,5.9,18.8
9,GOLD (true labels),,29.4,35.4,22.0,6.3,7.0,0.0


In [14]:
# For posts each classifier gets RIGHT in Chinese,
# how often does it still get them right in English?
# (This removes posts the classifier can't handle even in Chinese.)

for classifier_name in classifiers_run:
    chinese_column = classifier_name + " | Chinese original"
    correct_in_chinese = hadqaet[hadqaet[chinese_column] == hadqaet["gold"]]
    print(classifier_name)
    print("   Posts correct in Chinese:", len(correct_in_chinese), "of", len(hadqaet))

    for version_name in ["Google MT English", "Human-quality English"]:
        prediction_column = classifier_name + " | " + version_name
        still_correct = (correct_in_chinese[prediction_column] == correct_in_chinese["gold"]).mean()
        print("  ", version_name, "-> still correct:", round(still_correct * 100, 1), "%")

mBERT original (nb 04)
   Posts correct in Chinese: 3041 of 4038
   Google MT English -> still correct: 40.5 %
   Human-quality English -> still correct: 45.0 %
mBERT rebalanced (nb 04b)
   Posts correct in Chinese: 3024 of 4038
   Google MT English -> still correct: 59.8 %
   Human-quality English -> still correct: 64.6 %
XLM-R rebalanced (nb 04c)
   Posts correct in Chinese: 3189 of 4038
   Google MT English -> still correct: 65.1 %
   Human-quality English -> still correct: 69.7 %


In [15]:
# When a classifier does NOT predict neutral on English, how often is it right?
# (If this is close to the Chinese number, the English problem is mostly the neutral bias.)

for classifier_name in classifiers_run:
    print(classifier_name)
    for version_name in versions:
        prediction_column = classifier_name + " | " + version_name
        not_neutral = hadqaet[hadqaet[prediction_column] != "neutral"]
        accuracy_when_not_neutral = (not_neutral[prediction_column] == not_neutral["gold"]).mean()
        print("  ", version_name, "-> accuracy when not neutral:", round(accuracy_when_not_neutral * 100, 1), "%")

mBERT original (nb 04)
   Chinese original -> accuracy when not neutral: 78.7 %
   Google MT English -> accuracy when not neutral: 75.0 %
   Human-quality English -> accuracy when not neutral: 75.3 %
mBERT rebalanced (nb 04b)
   Chinese original -> accuracy when not neutral: 78.5 %
   Google MT English -> accuracy when not neutral: 66.7 %
   Human-quality English -> accuracy when not neutral: 68.6 %
XLM-R rebalanced (nb 04c)
   Chinese original -> accuracy when not neutral: 81.7 %
   Google MT English -> accuracy when not neutral: 72.6 %
   Human-quality English -> accuracy when not neutral: 73.9 %


In [16]:
# Save the per-post predictions and the summary table
os.makedirs(results_folder, exist_ok=True)

predictions_path = os.path.join(results_folder, "classifier_diagnostic_hadqaet_predictions.csv")
summary_path = os.path.join(results_folder, "classifier_diagnostic_summary.csv")

hadqaet.to_csv(predictions_path, index=False, encoding="utf-8-sig")
summary.to_csv(summary_path, index=False, encoding="utf-8-sig")

print("Saved:", predictions_path)
print("Saved:", summary_path)

Saved: /content/W266-slang-emotion-mt/data/results/classifier_diagnostic_hadqaet_predictions.csv
Saved: /content/W266-slang-emotion-mt/data/results/classifier_diagnostic_summary.csv
